In [3]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "Ramesh_StockMarket_Analytics_Lakehouse"
    }
}

StatementMeta(, 135825b8-7190-4439-8424-62a5de7ef362, -1, Finished, Available, Finished, True)

In [4]:
# ============================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ============================================================

import requests
import pandas as pd
import time
from datetime import datetime

# Companies to refresh
COMPANIES = ["IBM", "AAPL", "JPM", "AMZN", "KO"]

# Alpha Vantage API
API_KEY = "C3JXH6YRYNM0URHPKEY"
BASE_URL = "https://www.alphavantage.co/query"

print("Stock Market Daily Refresh")
print("=" * 50)
print("Companies:", COMPANIES)
print("Company count:", len(COMPANIES))
print("Configuration loaded successfully")

StatementMeta(, 135825b8-7190-4439-8424-62a5de7ef362, 3, Finished, Available, Finished, False)

Stock Market Daily Refresh
Companies: ['IBM', 'AAPL', 'JPM', 'AMZN', 'KO']
Company count: 5
Configuration loaded successfully


In [5]:
# ============================================================
# CELL 2 — LAKEHOUSE CONNECTION TEST
# ============================================================

stock_count = spark.table("stock_prices").count()
company_count = spark.table("dim_company").count()

print("Lakehouse connection test")
print("=" * 50)
print("stock_prices rows:", stock_count)
print("dim_company rows:", company_count)

if stock_count > 0 and company_count == 5:
    print("Lakehouse connection successful")
else:
    print("Lakehouse connection needs attention")

StatementMeta(, 135825b8-7190-4439-8424-62a5de7ef362, 4, Finished, Available, Finished, False)

Lakehouse connection test
stock_prices rows: 500
dim_company rows: 5
Lakehouse connection successful


In [6]:
# ============================================================
# CELL 3 — EXTRACT LATEST STOCK DATA
# ============================================================

def fetch_stock_data(symbol):
    print(f"Fetching data for {symbol}...")

    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "outputsize": "compact",
        "apikey": API_KEY
    }

    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    # Check for API errors or rate-limit messages
    if "Error Message" in data:
        raise Exception(f"{symbol}: {data['Error Message']}")

    if "Note" in data:
        raise Exception(f"{symbol}: Alpha Vantage API limit reached. {data['Note']}")

    if "Time Series (Daily)" not in data:
        raise Exception(
            f"{symbol}: Expected daily stock data was not returned. "
            f"Response keys: {list(data.keys())}"
        )

    time_series = data["Time Series (Daily)"]

    rows = []

    for date, values in time_series.items():
        rows.append({
            "Date": date,
            "Open": float(values["1. open"]),
            "High": float(values["2. high"]),
            "Low": float(values["3. low"]),
            "Close": float(values["4. close"]),
            "Volume": int(values["5. volume"]),
            "Symbol": symbol
        })

    df = pd.DataFrame(rows)

    # Convert date and sort oldest → newest
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

    # Calculate previous close and daily change
    df["Previous_Close"] = df["Close"].shift(1)

    df["Daily_Change_Pct"] = (
        (df["Close"] - df["Previous_Close"])
        / df["Previous_Close"]
        * 100
    )

    # First row has no previous trading day
    df["Previous_Close"] = df["Previous_Close"].fillna(df["Close"])
    df["Daily_Change_Pct"] = df["Daily_Change_Pct"].fillna(0)

    print(f"{symbol}: {len(df)} records extracted")

    return df


# Extract all companies
all_data = []

for i, symbol in enumerate(COMPANIES):

    df_company = fetch_stock_data(symbol)
    all_data.append(df_company)

    # Wait between API requests
    if i < len(COMPANIES) - 1:
        print("Waiting 3 seconds before next request...")
        time.sleep(3)


# Combine all companies
df_all = pd.concat(all_data, ignore_index=True)

# Sort final dataset
df_all = df_all.sort_values(
    ["Date", "Symbol"]
).reset_index(drop=True)

print()
print("=" * 60)
print("ALL COMPANIES EXTRACTED SUCCESSFULLY")
print("=" * 60)

print("Total rows:", len(df_all))
print("Companies:", sorted(df_all["Symbol"].unique()))

print()
print("Rows per company:")
print(df_all["Symbol"].value_counts().sort_index())

StatementMeta(, 135825b8-7190-4439-8424-62a5de7ef362, 5, Finished, Available, Finished, False)

Fetching data for IBM...
IBM: 100 records extracted
Waiting 3 seconds before next request...
Fetching data for AAPL...
AAPL: 100 records extracted
Waiting 3 seconds before next request...
Fetching data for JPM...
JPM: 100 records extracted
Waiting 3 seconds before next request...
Fetching data for AMZN...
AMZN: 100 records extracted
Waiting 3 seconds before next request...
Fetching data for KO...
KO: 100 records extracted

ALL COMPANIES EXTRACTED SUCCESSFULLY
Total rows: 500
Companies: ['AAPL', 'AMZN', 'IBM', 'JPM', 'KO']

Rows per company:
Symbol
AAPL    100
AMZN    100
IBM     100
JPM     100
KO      100
Name: count, dtype: int64


In [7]:
# ============================================================
# CELL 4 — DATA QUALITY VALIDATION
# ============================================================

print("DATA QUALITY CHECK")
print("=" * 60)

# 1. Missing values
missing_values = df_all.isnull().sum()

print("\nMissing values:")
print(missing_values)

# 2. Duplicate Symbol + Date
duplicate_count = df_all.duplicated(
    subset=["Symbol", "Date"]
).sum()

print("\nDuplicate Symbol + Date records:", duplicate_count)

# 3. Invalid prices
invalid_price_count = (
    (df_all["Open"] <= 0) |
    (df_all["High"] <= 0) |
    (df_all["Low"] <= 0) |
    (df_all["Close"] <= 0)
).sum()

print("Invalid price records:", invalid_price_count)

# 4. Invalid OHLC relationships
invalid_ohlc_count = (
    (df_all["High"] < df_all["Low"]) |
    (df_all["High"] < df_all["Open"]) |
    (df_all["High"] < df_all["Close"]) |
    (df_all["Low"] > df_all["Open"]) |
    (df_all["Low"] > df_all["Close"])
).sum()

print("Invalid OHLC records:", invalid_ohlc_count)

# 5. Invalid volume
invalid_volume_count = (
    df_all["Volume"] <= 0
).sum()

print("Invalid volume records:", invalid_volume_count)

# 6. Company count
company_count = df_all["Symbol"].nunique()

# 7. Total rows
total_rows = len(df_all)

# 8. Date range
minimum_date = df_all["Date"].min()
maximum_date = df_all["Date"].max()

print("\nCompany count:", company_count)
print("Total rows:", total_rows)
print("Date range:", minimum_date.date(), "to", maximum_date.date())

# Final validation
if (
    missing_values.sum() == 0
    and duplicate_count == 0
    and invalid_price_count == 0
    and invalid_ohlc_count == 0
    and invalid_volume_count == 0
    and company_count == 5
):
    print("\n✅ DATA QUALITY CHECK PASSED")
else:
    raise Exception("❌ DATA QUALITY CHECK FAILED")

StatementMeta(, 135825b8-7190-4439-8424-62a5de7ef362, 6, Finished, Available, Finished, False)

DATA QUALITY CHECK

Missing values:
Date                0
Open                0
High                0
Low                 0
Close               0
Volume              0
Symbol              0
Previous_Close      0
Daily_Change_Pct    0
dtype: int64

Duplicate Symbol + Date records: 0
Invalid price records: 0
Invalid OHLC records: 0
Invalid volume records: 0

Company count: 5
Total rows: 500
Date range: 2026-05-14 to 2026-10-06

✅ DATA QUALITY CHECK PASSED


In [8]:
# ============================================================
# CELL 5 — INCREMENTAL LOAD CHECK
# ============================================================

existing_df = spark.table("stock_prices").select(
    "Symbol",
    "Date"
)

existing_keys = {
    (row["Symbol"], row["Date"])
    for row in existing_df.collect()
}

new_keys = {
    (row["Symbol"], row["Date"])
    for _, row in df_all.iterrows()
}

new_records = new_keys - existing_keys
existing_records = new_keys & existing_keys

print("INCREMENTAL LOAD CHECK")
print("=" * 60)
print("API records:", len(new_keys))
print("Existing matching records:", len(existing_records))
print("New records:", len(new_records))

StatementMeta(, 135825b8-7190-4439-8424-62a5de7ef362, 7, Finished, Available, Finished, False)

INCREMENTAL LOAD CHECK
API records: 500
Existing matching records: 495
New records: 5


In [9]:
# ============================================================
# CELL 6 — PRODUCTION UPSERT
# ============================================================

from delta.tables import DeltaTable

# Convert Pandas data to Spark DataFrame
df_spark = spark.createDataFrame(df_all)

# Get production Delta table
target = DeltaTable.forName(spark, "stock_prices")

# Perform UPSERT
(
    target.alias("target")
    .merge(
        df_spark.alias("source"),
        """
        target.Symbol = source.Symbol
        AND target.Date = source.Date
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("PRODUCTION UPSERT COMPLETED SUCCESSFULLY")

# Validate production table
production_count = spark.table("stock_prices").count()

print("Production row count:", production_count)

StatementMeta(, 135825b8-7190-4439-8424-62a5de7ef362, 8, Finished, Available, Finished, False)

PRODUCTION UPSERT COMPLETED SUCCESSFULLY
Production row count: 505


In [10]:
# ============================================================
# CELL 7 — SAVE REFRESH SNAPSHOT TO HISTORY
# ============================================================

from pyspark.sql.functions import current_timestamp

# Read current production data
current_stock_df = spark.table("stock_prices")

# Add snapshot timestamp
history_snapshot = current_stock_df.withColumn(
    "Snapshot_Timestamp",
    current_timestamp()
)

# Append the current production state to history
history_snapshot.write.format("delta").mode("append").saveAsTable(
    "stock_prices_history"
)

# Validate history
history_count = spark.table("stock_prices_history").count()

print("REFRESH SNAPSHOT SAVED SUCCESSFULLY")
print("Total history rows:", history_count)

StatementMeta(, 135825b8-7190-4439-8424-62a5de7ef362, 9, Finished, Available, Finished, False)

REFRESH SNAPSHOT SAVED SUCCESSFULLY
Total history rows: 1995


In [12]:
# ============================================================
# CELL 8 — FINAL REFRESH VALIDATION
# ============================================================

print("FINAL REFRESH VALIDATION")
print("=" * 60)

# Production table
production_df = spark.table("stock_prices")

production_rows = production_df.count()
production_companies = production_df.select("Symbol").distinct().count()

# Duplicate check
duplicate_rows = (
    production_df
    .groupBy("Symbol", "Date")
    .count()
    .filter("count > 1")
    .count()
)

# Date range
date_range = production_df.selectExpr(
    "MIN(Date) AS Minimum_Date",
    "MAX(Date) AS Maximum_Date"
).collect()[0]

# History table
history_rows = spark.table("stock_prices_history").count()

print("Production rows:", production_rows)
print("Production companies:", production_companies)
print("Production duplicate keys:", duplicate_rows)
print(
    "Production date range:",
    date_range["Minimum_Date"],
    "to",
    date_range["Maximum_Date"]
)
print("History rows:", history_rows)

# ============================================================
# DYNAMIC VALIDATION
# ============================================================

if (
    production_rows > 0
    and production_companies == 5
    and duplicate_rows == 0
    and history_rows >= production_rows
):
    print()
    print("✅ FINAL REFRESH VALIDATION PASSED")
else:
    raise Exception("❌ FINAL REFRESH VALIDATION FAILED")

StatementMeta(, 135825b8-7190-4439-8424-62a5de7ef362, 11, Finished, Available, Finished, False)

FINAL REFRESH VALIDATION
Production rows: 505
Production companies: 5
Production duplicate keys: 0
Production date range: 2026-05-13 00:00:00 to 2026-10-06 00:00:00
History rows: 1995

✅ FINAL REFRESH VALIDATION PASSED
